# 06 - Model comparison (YOLO11n vs YOLO11s) and deployment benchmark
Runtime: **T4 GPU**. Run the cells **one at a time**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

token = userdata.get('GH_TOKEN')
REPO = 'ThemistoklisSakkas/sar-ship-robustness'
DEST = '/content/sar-ship-robustness'
if os.path.isdir(DEST):
    !git -C {DEST} pull -q https://{token}@github.com/{REPO}.git main
else:
    !git clone -q https://{token}@github.com/{REPO}.git {DEST}
    !git -C {DEST} remote set-url origin https://github.com/{REPO}.git
%cd {DEST}
!pip install -q -r requirements.txt

DRIVE_ROOT = '/content/drive/MyDrive/sar-ship-robustness'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_ROOT = f'{DRIVE_ROOT}/runs'
NAME_N = 'robust_yolo11n'
NAME_S = 'robust_yolo11s'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Copy the dataset to the local disk (1-2 min, once per runtime)

In [ ]:
LOCAL = '/content/data_local'
!mkdir -p {LOCAL} && cp -r {DATA_ROOT}/ssdd_yolo {LOCAL}/
!sed -i "s#^path: .*#path: {LOCAL}/ssdd_yolo#" {LOCAL}/ssdd_yolo/*.yaml
!rm -f {LOCAL}/ssdd_yolo/labels/*.cache
!grep "^path" {LOCAL}/ssdd_yolo/ssdd.yaml

## Step 1 - train YOLO11s with the phase-5 robust recipe (about 60-75 min)
After a disconnect: re-run Drive, setup and local copy, then the resume cell.

In [ ]:
!python -m sarship.train --data-root {LOCAL} --runs-root {RUNS_ROOT} --name {NAME_S} --set model.weights=yolo11s.pt robust_train.enabled=true

In [ ]:
# Only after a disconnect:
# !python -m sarship.train --data-root {LOCAL} --runs-root {RUNS_ROOT} --name {NAME_S} --resume

## Step 2 - evaluate YOLO11s, robustness study, paired comparison with YOLO11n
The comparison prints a "verdict" line from the phase-5 rule (clean / degraded); for
model selection use the phase-6 rule in RESULTS.md (gain >= 0.01 with CI above 0).

In [ ]:
!python -m sarship.evaluate --weights {RUNS_ROOT}/{NAME_S}/weights/best.pt --data-root {LOCAL} --runs-root {RUNS_ROOT} --name {NAME_S}

In [ ]:
!python -m sarship.robustness --weights {RUNS_ROOT}/{NAME_S}/weights/best.pt --data-root {LOCAL} --runs-root {RUNS_ROOT} --name {NAME_S} --work-dir /content/degraded_s

In [ ]:
!python -m sarship.compare --data-root {LOCAL} --runs-root {RUNS_ROOT} --set compare.baseline={NAME_N} compare.candidate={NAME_S}

## Step 3 - deployment benchmark: ONNX export, equivalence check, latency (about 10 min)

In [ ]:
!python -m sarship.benchmark --data-root {LOCAL} --runs-root {RUNS_ROOT} --names {NAME_N} {NAME_S}

In [ ]:
from IPython.display import Image, display
for path in [f'{RUNS_ROOT}/{NAME_S}/evaluation/training_curves.png',
             f'{RUNS_ROOT}/{NAME_S}/comparison/comparison.png',
             f'{RUNS_ROOT}/benchmark/accuracy_vs_speed.png']:
    print(path.split('/')[-1])
    if os.path.exists(path):
        display(Image(path))